In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test_x.csv")
sample = pd.read_csv("../data/sample_submission.csv")

In [3]:
target = "bilissel_performans_skoru"
y = train[target]

train = train.drop(columns=[target, "id"])
test_id = test["id"]
test = test.drop(columns=["id"])

df = pd.concat([train, test], axis=0).reset_index(drop=True)

In [ ]:
# Eksik Değerleri Doldurma
cat_features = df.select_dtypes(include=["object"]).columns.tolist()
for col in cat_features:
    df[col] = df[col].fillna("missing")

# Yeni Özellikler Üretme
df['kaliteli_uyku_toplam'] = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
df['kafein_ekran_etkisi'] = df['uyku_oncesi_kafein_mg'] * df['uyku_oncesi_ekran_suresi_dk']
df['stres_calisma_orani'] = df['stres_skoru'] * df['gunluk_calisma_saati']
df['haftalik_uyku_farki_etkisi'] = df['hafta_sonu_uyku_farki_saat'] * df['uykuya_dalma_suresi_dk']

# Gruplama İstatistikleri (Target/Mean Encoding alternatifleri)
for col in ['meslek', 'ulke', 'ruh_sagligi_durumu']:
    df[f'{col}_stres_mean'] = df.groupby(col)['stres_skoru'].transform('mean')
    df[f'{col}_calisma_mean'] = df.groupby(col)['gunluk_calisma_saati'].transform('mean')

In [ ]:
for col in cat_features:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

    df[col] = df[col].astype('category')

In [ ]:
X_train = df.iloc[:len(train)].copy()
X_test = df.iloc[len(train):].copy()

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds_cat = np.zeros(len(X_train))
oof_preds_lgb = np.zeros(len(X_train))
oof_preds_xgb = np.zeros(len(X_train))

test_preds_cat = np.zeros(len(X_test))
test_preds_lgb = np.zeros(len(X_test))
test_preds_xgb = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train, y)):
    print(f"\n{'='*20}\n--- FOLD {fold+1} ---\n{'='*20}")
    
    X_tr, y_tr = X_train.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X_train.iloc[valid_idx], y.iloc[valid_idx]
    
    # --- 1. CatBoost ---
    print("Eğitiliyor: CatBoost...")
    model_cat = CatBoostRegressor(
        iterations=1500, learning_rate=0.03, depth=6,
        loss_function="RMSE", random_seed=42,
        early_stopping_rounds=100, verbose=False
    )
    model_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va), cat_features=cat_features, use_best_model=True)
    oof_preds_cat[valid_idx] = model_cat.predict(X_va)
    test_preds_cat += model_cat.predict(X_test) / kf.n_splits
    
    # --- 2. LightGBM ---
    print("Eğitiliyor: LightGBM...")
    model_lgb = lgb.LGBMRegressor(
        n_estimators=1500, learning_rate=0.03, max_depth=6,
        random_state=42, n_jobs=-1, verbose=-1
    )
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)], 
        eval_metric='rmse',
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    oof_preds_lgb[valid_idx] = model_lgb.predict(X_va)
    test_preds_lgb += model_lgb.predict(X_test) / kf.n_splits
    
    # --- 3. XGBoost ---
    print("Eğitiliyor: XGBoost...")
    model_xgb = xgb.XGBRegressor(
        n_estimators=1500, learning_rate=0.03, max_depth=6,
        random_state=42, n_jobs=-1, enable_categorical=True,
        early_stopping_rounds=100
    )
    model_xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )
    oof_preds_xgb[valid_idx] = model_xgb.predict(X_va)
    test_preds_xgb += model_xgb.predict(X_test) / kf.n_splits



--- FOLD 1 ---
Eğitiliyor: CatBoost...
Eğitiliyor: LightGBM...
Eğitiliyor: XGBoost...

--- FOLD 2 ---
Eğitiliyor: CatBoost...
Eğitiliyor: LightGBM...
Eğitiliyor: XGBoost...

--- FOLD 3 ---
Eğitiliyor: CatBoost...
Eğitiliyor: LightGBM...
Eğitiliyor: XGBoost...

--- FOLD 4 ---
Eğitiliyor: CatBoost...
Eğitiliyor: LightGBM...
Eğitiliyor: XGBoost...

--- FOLD 5 ---
Eğitiliyor: CatBoost...
Eğitiliyor: LightGBM...
Eğitiliyor: XGBoost...


In [ ]:
rmse_cat = np.sqrt(mean_squared_error(y, oof_preds_cat))
rmse_lgb = np.sqrt(mean_squared_error(y, oof_preds_lgb))
rmse_xgb = np.sqrt(mean_squared_error(y, oof_preds_xgb))

print(f"CatBoost OOF RMSE: {rmse_cat:.4f}")
print(f"LightGBM OOF RMSE: {rmse_lgb:.4f}")
print(f"XGBoost OOF RMSE : {rmse_xgb:.4f}")

oof_preds_ens = (oof_preds_cat + oof_preds_lgb + oof_preds_xgb) / 3
rmse_ens = np.sqrt(mean_squared_error(y, oof_preds_ens))
print(f"\n--> Ensemble OOF RMSE: {rmse_ens:.4f}")

CatBoost OOF RMSE: 1.2162
LightGBM OOF RMSE: 1.2255
XGBoost OOF RMSE : 1.2275

--> Ensemble OOF RMSE: 1.2191


In [ ]:
test_preds_ens = (test_preds_cat + test_preds_lgb + test_preds_xgb) / 3
test_preds_ens = test_preds_ens.clip(0, 10)

submission = pd.DataFrame({
    "id": test_id,
    "bilissel_performans_skoru": test_preds_ens
})

submission.to_csv("../submissions/sub_advanced.csv", index=False)
submission.head()

,id,bilissel_performans_skoru
0,1,5.899028
1,2,6.691026
2,3,3.013149
3,4,7.057880
4,5,3.676828
